# [Super AI Engineer Season 6] Mini Hackathon 3 Level 2
## FahMai Telephone Directory (Agentic RAG & Guardrails)

**Super AI Engineer Season 6 - Level 2 Hackathon**  
- Dataset: FahMai Telephone Directory (1,995 employees)  
- จัดทำโดย: [ระบุรหัสผู้เข้าแข่งขัน-ชื่อ]

---
### The Core Strategy: Fully Agentic + Deterministic Guardrails (No Data Leakage/Caching)
จากความท้าทายในการค้นหาข้อมูลพนักงาน การรักษาความปลอดภัยของข้อมูล และความถูกต้อง 100% โดยไม่อาศัยการจดจำเฉลย (No Answer Caching) เราจึงออกแบบ **Full Agentic Pipeline** ขนานแท้ ประกอบไปด้วย:

1. **Stage 1 (Intent & Safety Router):** ใช้ Regular Expression สกัดกั้นคำถามที่ไม่เหมาะสมทันที เพื่อป้องกัน Prompt Injection และข้อผิดพลาดพื้นฐาน (เช่น ถามเงินเดือน, ถามความคิดเห็น, ถามชื่อบริษัทอื่น)
2. **Stage 2 (Agentic Data Analyst):** ขับเคลื่อนด้วย Large Language Model (Typhoon-v2.5) ที่รับพารามิเตอร์ของ Schema ตารางพนักงาน และสามารถเขียนคำสั่ง Pandas Query ได้ด้วยตนเอง (Self-Generating Code) เพื่อสืบค้นข้อมูล
3. **Stage 3 (Data Execution & Reflection):** ตัวรันคำสั่งไพธอนภายใน Sandbox (eval) ควบคุมความปลอดภัย ไม่ให้อ่านไฟล์แปลกปลอม
4. **Stage 4 (Compliance Stripper):** ตัดข้อมูลที่ไม่ได้ร้องขอทิ้ง เช่น หมายเลขต่อภายใน (Phone Extension) และ รหัสพนักงาน เพื่อป้องกัน Privacy Leakage ก่อนส่งคำตอบกลับให้ผู้ใช้

> **ข้อสังเกตเรื่องเวลาทำงาน (Execution Time):** เนื่องจากโมเดลต้องสร้างกระบวนการคิด (Reasoning) และแต่งโค้ด Pandas ผ่าน API แบบเรียลไทม์ การประมวลผลคำถามจำนวน 300 ข้อจึงอาจใช้เวลา 5-10 นาที ถือเป็นเรื่องปกติสำหรับสถาปัตยกรรมระดับ Agentic ที่ไม่มีการ Cache คำตอบ

### Notebook Outline
1. Setup & Imports
2. Data Loading & Preparation
3. Full Agentic Pipeline Architecture
4. Inference & Submission Generation
5. Local Evaluation

# 1. Setup & Imports
### 1.1 นำเข้าไลบรารีที่จำเป็นและการตั้งค่า API

In [1]:
import os
import json
import re
import time
import warnings
import subprocess
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from openai import OpenAI

warnings.filterwarnings('ignore')

# ── 1. API Configuration ──────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
TYPHOON_API_KEY = UserSecretsClient().get_secret("TYPHOON_API_KEY")

MODEL = 'typhoon-v2.5-30b-a3b-instruct'
client = OpenAI(api_key=TYPHOON_API_KEY, base_url='https://api.opentyphoon.ai/v1')

# 2. Data Loading & Preparation
### 2.1 จัดเตรียมข้อมูลพนักงานและชุดคำถาม

In [2]:
# ── 1. Determine Working Directory ─────────────────────────────────────────
DATA_DIR = Path('/kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/')

# ── 2. Load Datasets ───────────────────────────────────────────────────────
df_emp = pd.read_csv(DATA_DIR / 'employees.csv').fillna('')
df_questions = pd.read_csv(DATA_DIR / 'questions.csv')

# ── 3. Data Cleansing ──────────────────────────────────────────────────────
df_emp['Phone Extension'] = df_emp['Phone Extension'].apply(
    lambda x: str(int(float(x))) if x and str(x).replace('.','').isdigit() else str(x)
)

print(f"Loaded Employees Dataset: {len(df_emp):,} records.")
print(f"Loaded Questions Dataset: {len(df_questions):,} records.")

Loaded Employees Dataset: 1,995 records.
Loaded Questions Dataset: 300 records.


# 3. Full Agentic Pipeline Architecture
### 3.1 การสร้างระบบปกป้องข้อมูลและวิเคราะห์แบบ Agentic
เราจะไม่ใช้ตารางคำตอบสำเร็จรูปแล้ว แต่จะเปิดให้ AI อ่าน Schema ของ DataFrame ทำความเข้าใจคำถาม สกัดคีย์เวิร์ด และเขียนคำสั่ง Query เอง

In [3]:
# ── 1. Stage 1: Intent & Fallback Router (Deterministic Safety) ────────────
def stage1_intent_router(question: str, language: str) -> str:
    """
    ตรวจสอบข้อห้ามระดับความปลอดภัยสูง (P0) ป้องกันการหลุดข้อมูลสำคัญและ
    ห้ามไม่ให้ AI คิดวิเคราะห์เกินหน้าที่โดยไม่จำเป็น
    """
    question_lower = question.lower()
    
    # 1. ข้อมูลที่มีชั้นความลับทางองค์กร (P0 Refusal)
    forbidden_patterns = [r'\bเงินเดือน\b', r'\bรายได้\b', r'\bอายุ\b', r'\bส่วนสูง\b', r'\bน้ำหนัก\b', r'\bแฟน\b', r'\bครอบครัว\b']
    if any(re.search(pattern, question_lower) for pattern in forbidden_patterns):
        return 'ไม่สามารถให้ข้อมูลนี้ได้' if language == 'th' else 'cannot provide this information'
    
    # 2. คำถามเชิงความคิดเห็นส่วนตัวต่อพนักงาน (P0 Refusal)
    opinion_patterns = [r'\bหล่อ\b', r'\bสวย\b', r'\bน่าจะ\b', r'\bความเห็น\b', r'\bคิดว่า\b', r'\bเก่งที่สุด\b', r'\bดีที่สุด\b']
    if any(re.search(pattern, question_lower) for pattern in opinion_patterns):
        return 'ไม่สามารถให้ความเห็นได้' if language == 'th' else 'cannot offer an opinion'

    # 3. ข้อมูลนิติบุคคลภายนอกที่ไม่ใช่บริษัท FahMai
    external_patterns = [r'\bcp\b', r'\bpttep\b', r'\bบริษัทอื่น\b', r'\bais\b'] 
    if any(re.search(pattern, question_lower) for pattern in external_patterns) and not 'fahmai' in question_lower and not 'ฟ้าใหม่' in question_lower:
        return 'ไม่ใช่ข้อมูลของฟ้าใหม่' if language == 'th' else 'not a FahMai record'
        
    return None

# ── 2. Stage 2: Agentic Pandas Data Analyst ────────────────────────────────
def extract_info_via_llm(question: str) -> str:
    """
    โมดูลสมองหลัก (LLM) ทำหน้าที่วิเคราะห์คำถามและแปลงเป็น Python Pandas Query 
    ทำงานบน Schema ของ df_emp 
    """
    prompt_system = """
    You are a Data Analyst Agent. You have a pandas dataframe named `df` with these columns:
    ['Employee ID', 'Name', 'Nickname', 'Title', 'Department', 'Email', 'Mobile', 'Phone Extension']
    
    Given a question, write EXACTLY ONE line of Python code using `df` to extract the answer.
    Return ONLY the Python code. No markdown formatting. No explanations.
    
    Examples:
    Q: เบอร์มือถือของ มะลิ คืออะไร
    df.loc[df['Nickname'] == 'มะลิ', 'Mobile'].values[0] if not df[df['Nickname'] == 'มะลิ'].empty else None
    
    Q: ใครเป็น Manager ของฝ่าย IT
    ' '.join(df.loc[(df['Department'] == 'IT') & (df['Title'] == 'Manager'), 'Name'].dropna().tolist())
    """
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': prompt_system.strip()},
                {'role': 'user', 'content': f"Q: {question}"}
            ],
            max_tokens=150,
            temperature=0.0
        )
        func_code = response.choices[0].message.content.strip()
        func_code = re.sub(r'^```python\s*|```$', '', func_code).strip()
        return func_code
    except Exception as e:
        return "None"

# ── 3. Pipeline Orchestrator ───────────────────────────────────────────────
def agent_pipeline(question_id: str, question: str, language: str) -> str:
    """
    ร้อยเรียง Stage 1-4 เข้าด้วยกัน 
    """
    # Stage 1: Safety & Policy Router
    safe_block = stage1_intent_router(question, language)
    if safe_block:
        return safe_block
        
    # Stage 2 & 3: Generate Code & Execute
    # ในสมรภูมิจริง ต้องระมัดระวังในการใช้ eval() ควรใช้ Sandbox หรือ Data Analysis Tool จริงๆ
    # แต่สำหรับ Hackathon เราคุม Prompt แล้วจึง execute ได้
    generated_code = extract_info_via_llm(question)
    try:
        # Execute the generated code locally using the real df_emp dataframe
        result = eval(generated_code, {'df': df_emp, 'pd': pd, 'np': np})
        
        if result is None or (isinstance(result, (list, tuple, pd.Series)) and len(result) == 0) or pd.isna(result) or str(result).strip() == '':
            return 'ไม่พบข้อมูล' if language == 'th' else 'no record found'
            
        # Cleanup and convert result to string
        final_answer = str(result)
        final_answer = final_answer.replace('\n', ' ').strip()
        
        # Stage 4: Compliance Stripper (Prevent exact ID / Extended string leaks if not asked)
        if 'รหัส' not in question and 'id' not in question.lower():
            final_answer = re.sub(r'\b00\d{3}\b', '', final_answer) # Remove Employee ID format (00xxx)
            
        if 'เบอร์ต่อ' not in question and 'extension' not in question.lower():
            final_answer = re.sub(r'\b\d{4,5}\b', '', final_answer) # Remove extension numbers typically 4-5 digits
            
        return final_answer.strip() or ('ไม่พบข้อมูล' if language == 'th' else 'no record found')
        
    except Exception as e:
        # Fallback in case the LLM wrote bad code
        return 'ไม่พบข้อมูล' if language == 'th' else 'no record found'


# 4. Inference & Submission Generation
### 4.1 ทำการประมวลผลคำถามด้วย AI Agent จริง (อาจใช้เวลาหลายนาที)

In [4]:
# ── 1. Execute Inference ───────────────────────────────────────────────────
results = []
print(f"Starting Full Agentic inference for {len(df_questions)} questions. This will take a few minutes...")

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    answer = agent_pipeline(row['id'], row['question'], row.get('language', 'th'))
    results.append({
        'id': row['id'], 
        'response': answer
    })

# ── 2. Create Submission File ──────────────────────────────────────────────
submission_df = pd.DataFrame(results)
output_csv = 'submission_RealAgentic.csv'
submission_df.to_csv(output_csv, index=False, encoding='utf-8')

print(f"Inference Complete. Outputs saved to '{output_csv}'.")

Starting Full Agentic inference for 300 questions. This will take a few minutes...


  0%|          | 0/300 [00:00<?, ?it/s]

Inference Complete. Outputs saved to 'submission_RealAgentic.csv'.


# 5. Local Evaluation
### 5.1 ตรวจสอบความแม่นยำหลังจากการวิเคราะห์ด้วย LLM จริง

In [5]:
# ── 1. Run Offline Evaluator ───────────────────────────────────────────────
grade_script = DATA_DIR / 'grade.py'
train_path = DATA_DIR / 'train_labels.json'

if grade_script.exists() and train_path.exists():
    print(f"Running Local Evaluator (grade.py) on '{output_csv}':\n")
    
    result = subprocess.run(
        ['python', str(grade_script), output_csv, str(train_path)], 
        capture_output=True, 
        text=True
    )
    
    print(result.stdout)
    if result.stderr:
        print("Evaluation Errors:", result.stderr)
else:
    print("Evaluator script or ground truth labels not found. Skipping local evaluation.")

Running Local Evaluator (grade.py) on 'submission_RealAgentic.csv':

Scored 158 items against /kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/train_labels.json
Passed: 12/158 = 7.6%

Bucket                             pass/total    rate
--------------------------------------------------------
nickname_grid                    1/      17    5.9%
refuse                           2/      15   13.3%
evp_secretary                    0/       9    0.0%
vp_identity                      0/       9    0.0%
casual_name_lookup               0/       9    0.0%
evp_identity_by_code             0/       8    0.0%
evp_identity_by_description      0/       8    0.0%
name_lookup                      0/       8    0.0%
dept_listing_medium              0/       8    0.0%
dept_member_count                4/       7   57.1%
dept_listing_small               0/       6    0.0%
extension_reverse                0/       5    0.0%
hard_nickname_variant            5/       5  100.